# ETP Semantic Drift Experiment — Core Setup

This notebook is structured around the **5-component experimental setting**:

1. **Intended theory** — an equation from ETP
2. **Representation change** — one LLM call transforms it into another form
3. **Generated output** — classify what the model produced (valid? which drift type?)
4. **Semantic validation** — compare against the intended equation using 2+ methods
5. **Analysis** — summarize where drift happened and what each method caught

We build this **incrementally**: Section A gets the single LLM call working and verified in isolation before anything else depends on it. Only after that works do we chain it into the full pipeline.


## Setup: Drive, dependencies, API key

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/etp_pipeline'
DRIVE_DB_PATH = '/content/drive/MyDrive/etp_pipeline/implications.db'
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_DB_PATH), exist_ok=True)
%cd $WORKDIR


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/etp_pipeline


In [ ]:
!pip install -q transformers accelerate sympy


## Write the oracle source files (needed before Section A)

In [ ]:
%%writefile find_equation_id.py
#!/usr/bin/env python3

"""This module maps magma equations from/to their id

It can be used as a script in interactive mode (with the -i switch), as

    python find_equation_id.py -i

or by passing arguments to it from stdin or as arguments: a (space-separated)
list of ids or of equations (in which the operation can be ".", "*", or "◇"),
optionally preceded by "*" to dualize the equation (and characters "[,]" are
ignored):

    python find_equation_id.py [12, 34] "(w*u)=t*(u*x)" 4567 "*89" "*67" "0=(1*2)*(0*1)"

When used as a module imported in python code, one can use
- eq = Equation.from_id(integer id)
- eq = Equation.from_str(string)
- eq.id
- all_eqs(integer order)
- eq.dual()

The theory of magma operations and their labeling is explained in
https://teorth.github.io/equational_theories/blueprint/basic-theory-chapter.html

"""

import sys
import argparse
import itertools
import typing
import functools
from sympy.functions.combinatorial.numbers import bell, binomial, catalan
import math
import string

VAR_NAMES = "xyzwuvrst"

ExprType = typing.Union[str, int, typing.Tuple["ExprType", str, "ExprType"]]
ShapeType = typing.Union[None, typing.Tuple["ShapeType", "ShapeType"]]


class Equation(typing.NamedTuple):
    """Equation(lhs_shape, rhs_shape, rhyme) denotes an equation

    lhs_shape and rhs_shape are nested pairs (tuples) of None giving how
    the operation is nested, and rhyme a tuple of int (starting with 0)
    giving the rhyme scheme (variable names, as numbers).  For instance,
    Equation(None, ((None, None), None), (0, 1, 0, 2)) is x=(y*x)*z.
    """
    lhs_shape: ShapeType
    rhs_shape: ShapeType
    rhyme: typing.Tuple[int]

    @classmethod
    def from_id(cls, eq_id: int) -> "Equation":
        """Construct an equation given its id."""
        return _equation_from_id(eq_id)

    @property
    def id(self) -> int:
        """Evaluate the id of the equation."""
        return _equation_id(self)

    @classmethod
    def from_str(cls, eq_str: str) -> "Equation":
        """Parse and canonicalize an equation given as a string."""
        return _equation_from_str(eq_str)

    def __str__(self) -> str:
        rhyme_iter = iter(self.rhyme)
        lhs_str = Equation._expr_str(self.lhs_shape, rhyme_iter, False)
        rhs_str = Equation._expr_str(self.rhs_shape, rhyme_iter, False)
        return f"{lhs_str} = {rhs_str}"

    @classmethod
    def _expr_str(cls, shape: ShapeType, rhyme_iter: typing.Iterator[int], parenthesize: bool) -> str:
        if shape is None:
            i, j = divmod(next(rhyme_iter), len(VAR_NAMES))
            if i == 0:
                return VAR_NAMES[j]
            return VAR_NAMES[j] + str(i)
        left_str = cls._expr_str(shape[0], rhyme_iter, True)
        right_str = cls._expr_str(shape[1], rhyme_iter, True)
        if parenthesize:
            return f"({left_str} ◇ {right_str})"
        return f"{left_str} ◇ {right_str}"

    def orders(self) -> typing.Tuple[int, int]:
        """Number of operations on the lhs and rhs as a tuple."""
        return (shape_order(self.lhs_shape), shape_order(self.rhs_shape))

    def num_vars(self) -> int:
        """Number of distinct variables in the equation."""
        return max(self.rhyme) + 1

    def dual(self) -> "Equation":
        """Swap all left and right operands, swap lhs and rhs if needed."""
        lhs_shape = shape_dual(self.lhs_shape)
        rhs_shape = shape_dual(self.rhs_shape)
        lhs_order = shape_order(self.lhs_shape)
        lhs_rhyme = tuple(reversed(self.rhyme[:lhs_order + 1]))
        rhs_rhyme = tuple(reversed(self.rhyme[lhs_order + 1:]))
        if shape_lt(rhs_shape, lhs_shape):
            lhs_shape, rhs_shape = rhs_shape, lhs_shape
            lhs_rhyme, rhs_rhyme = rhs_rhyme, lhs_rhyme
        rhyme = canonicalize_rhyme(lhs_rhyme + rhs_rhyme)
        if lhs_shape == rhs_shape:
            rhyme = min(rhyme, canonicalize_rhyme(rhs_rhyme + lhs_rhyme))
        return Equation(lhs_shape, rhs_shape, rhyme)

##### Parsing an equation string

def _tokenize(expr: str) -> typing.List[str]:
    """Convert an expression string into a list of tokens."""
    expr = (
        expr.replace(".", "◇")
        .replace("*", "◇")
        .replace("(", " ( ")
        .replace(")", " ) ")
        .replace("◇", " ◇ ")
    )
    return [token for token in expr.split() if token]


def _parse_expr(tokens: typing.List[str]) -> ExprType:
    """Parse a list of tokens into an expression tree.

    Return nested triplets (left, "◇", right) with variables as str or int."""

    def parse_element() -> ExprType:
        if not tokens:
            raise ValueError("Unexpected end of expression")
        if tokens[0] == "(":
            tokens.pop(0)  # Remove opening parenthesis
            left = parse_element()
            if not tokens or tokens[0] != "◇":
                raise ValueError("Expected '◇' after element in parentheses")
            tokens.pop(0)  # Remove '◇'
            right = parse_element()
            if not tokens or tokens[0] != ")":
                raise ValueError("Missing closing parenthesis")
            tokens.pop(0)  # Remove closing parenthesis
            return (left, "◇", right)
        if (tokens[0].isidentifier() or tokens[0] == "0" or
            (tokens[0][0] in "123456789" and
             all(c in "0123456789" for c in tokens[0][1:]))):
            return tokens.pop(0)
        raise ValueError(f"Unexpected token: {tokens[0]}")

    result = parse_element()
    if tokens:
        if tokens[0] != "◇":
            raise ValueError(f"Unexpected token after main element: {tokens[0]}")
        tokens.pop(0)  # Remove '◇'
        right = parse_element()
        if tokens:
            raise ValueError(
                f"Unexpected tokens at the end of expression: {' '.join(tokens)}"
            )
        result = (result, "◇", right)
    return result


def _deconstruct_tree(tree: ExprType) -> typing.Tuple[ShapeType, typing.List[str]]:
    if isinstance(tree, str):
        return (None, [tree])
    left, _op, right = tree
    left_shape, left_rhyme = _deconstruct_tree(left)
    right_shape, right_rhyme = _deconstruct_tree(right)
    return ((left_shape, right_shape), left_rhyme + right_rhyme)


def _equation_from_str(eq_str: str) -> Equation:
    try:
        lhs, rhs = eq_str.split("=")
    except ValueError:
        raise ValueError("No '=' or two '=' found in the equation.")
    lhs = _parse_expr(_tokenize(lhs))
    rhs = _parse_expr(_tokenize(rhs))
    lhs_shape, lhs_rhyme = _deconstruct_tree(lhs)
    rhs_shape, rhs_rhyme = _deconstruct_tree(rhs)
    if shape_lt(rhs_shape, lhs_shape):
        lhs_shape, rhs_shape = rhs_shape, lhs_shape
        lhs_rhyme, rhs_rhyme = rhs_rhyme, lhs_rhyme
    rhyme = canonicalize_rhyme(lhs_rhyme + rhs_rhyme)
    if lhs_shape == rhs_shape:
        rhyme = min(rhyme, canonicalize_rhyme(rhs_rhyme + lhs_rhyme))
    return Equation(lhs_shape, rhs_shape, rhyme)


##### On shapes

def shape_dual(shape: ShapeType) -> ShapeType:
    if shape is None:
        return None
    return (shape_dual(shape[1]), shape_dual(shape[0]))

def shape_order(shape: ShapeType) -> int:
    if shape is None:
        return 0
    return 1 + shape_order(shape[0]) + shape_order(shape[1])


def shape_cmp(shape1: ShapeType, shape2: ShapeType) -> int:
    shape1_order = shape_order(shape1)
    shape2_order = shape_order(shape2)
    if shape1_order < shape2_order:
        return -1
    if shape1_order > shape2_order:
        return 1
    if shape1 is None and shape2 is None:
        return 0
    left_cmp = shape_cmp(shape1[0], shape2[0])
    if left_cmp != 0:
        return left_cmp
    return shape_cmp(shape1[1], shape2[1])


def shape_lt(shape1: ShapeType, shape2: ShapeType) -> bool:
    return shape_cmp(shape1, shape2) < 0


##### Generating all rhymes, all shapes, all equations

def canonicalize_rhyme(rhyme: typing.Tuple[int]) -> typing.Tuple[int]:
    """Canonicalize the rhyme to increasing order."""
    variables = {}
    for x in rhyme:
        if x not in variables:
            variables[x] = len(variables)
    return tuple(variables[x] for x in rhyme)


def all_rhymes(n: int) -> typing.Iterator[typing.Tuple[int]]:
    """Generate all rhymes of a given length."""
    if n == 0:
        yield ()
        return
    for next in _all_rhymes_help(n, 0):
        yield (0,) + next


def _all_rhymes_help(n: int, max_used: int) -> typing.Iterator[typing.Tuple[int]]:
    """Generates all rhymes whose minimum is at most max_used + 1"""
    if n == 0:
        yield ()
        return
    for x in range(max_used + 2):
        for next in _all_rhymes_help(n - 1, max(max_used, x)):
            yield (x,) + next


def all_shapes(order: int) -> typing.Iterator[ShapeType]:
    """Generate all possible shapes for expressions with a given number of operations."""
    if order == 0:
        yield None
    for i in range(order):
        for left in all_shapes(i):
            for right in all_shapes(order - 1 - i):
                yield (left, right)


def all_eqs(order: int) -> typing.Iterator[Equation]:
    """Generate all unique equations of some order up to symmetry.

    To generate unique equations of all orders, use
    (eq for n in itertools.count() for eq in all_eqs(n)).
    """
    half = order // 2 + 1
    for lhs_order in range(half):
        for lhs_shape in all_shapes(lhs_order):
            for rhs_shape in all_shapes(order - lhs_order):
                if order == lhs_order * 2 and shape_lt(rhs_shape, lhs_shape):
                    continue
                symmetric_shape = lhs_shape == rhs_shape
                for rhyme in all_rhymes(order + 1):
                    if symmetric_shape:
                        flipped = rhyme[half:] + rhyme[:half]
                        if canonicalize_rhyme(flipped) < rhyme:
                            continue
                        if rhyme == flipped and order > 0:
                            continue
                    yield Equation(lhs_shape, rhs_shape, rhyme)


##### Recursive approach to mapping from equation number to id and vice-versa

# Counting equations of some order, based on https://oeis.org/A103293, refactored to access intermediate results.

@functools.lru_cache(maxsize=None)
def num_eqs(n: int) -> int:
    """Sequence https://oeis.org/A376640 of the number of magma equations"""
    if n % 2 == 1:
        return catalan(n + 1) * bell(n + 2) // 2
    else:
        if n == 0: return 2
        return ((catalan(n + 1) - catalan(n // 2)) * bell(n + 2) // 2
                + catalan(n // 2) * bell_same_shape(n))


@functools.lru_cache(maxsize=None)
def bell_same_shape(n: int) -> int:
    """Number of rhymes when lhs and rhs have the same (n//2)-operations shape"""
    if n == 0:
        return 2
    return (bell(n + 2) + sum(stirling_sym(n + 2, k) for k in range(n + 3))
            - 2 * bell(1 + n // 2)) // 2


@functools.lru_cache(maxsize=None)
def stirling_sym(n: int, k: int) -> int:
    """Number of symmetric k-partitions of range(n), see https://oeis.org/A103293"""
    if n < 2:
        return k == n
    return k * stirling_sym(n - 2, k) + stirling_sym(n - 2, k - 1) + stirling_sym(n - 2, k - 2)


# Map from shape to id and back

def shape_id(shape: ShapeType) -> int:
    """Gives the shape id (zero-based) among shapes of a given order"""
    return _shape_id_help(shape, shape_order(shape))


def _shape_id_help(shape: ShapeType, n: int) -> int:
    if n == 0:
        return 0
    lhs_shape, rhs_shape = shape
    lhs_n = shape_order(lhs_shape)
    rhs_n = n - 1 - lhs_n
    return (sum(catalan(n1) * catalan(n - n1 - 1) for n1 in range(lhs_n))
            + _shape_id_help(lhs_shape, lhs_n) * catalan(rhs_n)
            + _shape_id_help(rhs_shape, rhs_n))


def shape_from_id(nodes: int, tree_num: int) -> ShapeType:
    if nodes == 0:
        if tree_num != 0:
            raise ValueError
        return None
    for n1 in range(nodes):
        test_num = catalan(n1) * catalan(nodes - n1 - 1)
        if tree_num >= test_num:
            tree_num -= test_num
            continue
        tree_num_1, tree_num_2 = divmod(tree_num, catalan(nodes - n1 - 1))
        return (shape_from_id(n1, tree_num_1),
                shape_from_id(nodes - n1 - 1, tree_num_2))



# Map from rhyme to id and back

@functools.lru_cache(maxsize=None)
def _num_rhyme_help(n: int, max_used: int) -> int:
    """Number of rhymes of n slots whose minimum number is at most max_used + 1"""
    if n==0:
        return 1
    return (max_used + 1) * _num_rhyme_help(n - 1, max_used) + _num_rhyme_help(n - 1, max_used + 1)

def check_rhyme_id_is_canonical(p: typing.Tuple[int]) -> None:
    if (not p):
        raise ValueError("Argument of find_rhyme_id should be non-empty")
    next_used = 0
    for pi in p:
        if pi > next_used:
            raise ValueError(f"Argument of find_rhyme_id should have canonical form, not {p}")
        elif pi == next_used:
            next_used += 1

def find_rhyme_id(p: typing.Tuple[int]) -> int:
    """Gives the rhyme id (zero-based) among rhymes with a given number of variables"""
    check_rhyme_id_is_canonical(p)
    return _find_rhyme_id_help(p[1:], 0)


def _find_rhyme_id_help(p: typing.Tuple[int], max_used: int) -> int:
    return p[0] * _num_rhyme_help(len(p)-1, max_used) + _find_rhyme_id_help(p[1:], max(p[0], max_used)) if p else 0


def get_rhyme_by_id(n: int, rhyme_num: int, max_used: int = 0) -> typing.Tuple[int]:
    """Find a rhyme scheme for n slots by its number (zero-indexed)."""
    result = (0,)
    while n > 0:
        var1 = min(max_used + 1, rhyme_num // _num_rhyme_help(n - 1, max_used))
        result += (var1,)
        rhyme_num -= var1 * _num_rhyme_help(n - 1, max_used)
        max_used = max(max_used, var1)
        n -= 1
    return result


# Map from equation to id and back.

@functools.lru_cache(maxsize=None)
def _num_eqs_unbalanced(n: int) -> int:
    """Counts magma equations that have strictly fewer operations on the left than on the right"""
    return ((catalan(n + 1) - (0 if n % 2 == 1 else catalan(n // 2) ** 2))
            * bell(n + 2)) // 2


def _num_eqs_balanced(n: int, l: int, r: int) -> int:
    """Number of balanced equations before lhs/rhs shapes number l, r"""
    return (bell(n + 2) * (catalan(n // 2) * l - l * (l + 1) // 2
                           + r - l - (1 if r > l else 0))
            + bell_same_shape(n) * (l + (1 if r > l else 0)))


def _equation_id(input_eq: Equation) -> typing.Tuple[int, Equation]:
    """Equation id from a processed Equation"""
    lhs_shape = input_eq.lhs_shape
    rhs_shape = input_eq.rhs_shape
    n_lhs = shape_order(lhs_shape)
    n_rhs = shape_order(rhs_shape)
    n = n_lhs + n_rhs
    if n_lhs != n_rhs:
        return (1 + sum(num_eqs(i) for i in range(n))
                + bell(n + 2) * shape_id((lhs_shape, rhs_shape))
                + find_rhyme_id(input_eq.rhyme))
    # For n_lhs == n_rhs the ordering halves the equations.  For
    # different tree shapes get bell(n + 2) rhymes, otherwise
    # bell_same_shape(n).
    m = catalan(n_lhs) # number of tree shapes on each side
    l = shape_id(lhs_shape)
    r = shape_id(rhs_shape)
    if l != r:
        pid = find_rhyme_id(input_eq.rhyme)
    else:
        # Slow code here
        check_rhyme_id_is_canonical(input_eq.rhyme)
        pid = 0
        if n > 0 and input_eq.rhyme == input_eq.rhyme[n_lhs + 1:] + input_eq.rhyme[:n_lhs + 1]:
            return 0 # tautological equation
        for rhyme in all_rhymes(n + 1):
            if rhyme == input_eq.rhyme:
                break
            flipped = rhyme[n_lhs + 1:] + rhyme[:n_lhs + 1]
            if canonicalize_rhyme(flipped) < rhyme:
                continue
            if rhyme == flipped and n > 0:
                continue
            pid += 1
    return (1 + sum(num_eqs(i) for i in range(n))
            + _num_eqs_unbalanced(n) + _num_eqs_balanced(n, l, r)
            + pid)


def _equation_from_id(input_eq: int) -> Equation:
    n = 0
    eq_num = input_eq - 1
    while eq_num >= (max_eq_num := num_eqs(n)):
        eq_num -= max_eq_num
        n += 1
    if eq_num < _num_eqs_unbalanced(n):
        tree_num, rhyme_num = divmod(eq_num, bell(n + 2))
        lhs_shape, rhs_shape = shape_from_id(n + 1, tree_num)
        rhyme = get_rhyme_by_id(n + 1, rhyme_num)
        return Equation(lhs_shape, rhs_shape, rhyme)
    eq_num -= _num_eqs_unbalanced(n)
    m = catalan(n // 2)
    l = ((2*m - 1) * bell(n + 2) + 2 * bell_same_shape(n)
         - math.isqrt(((2 * m - 1) * bell(n + 2) + 2 * bell_same_shape(n)) ** 2
                 - 8 * bell(n + 2) * eq_num - 1)
         - 1) // (2*bell(n + 2))
    lhs_shape = shape_from_id(n // 2, l)
    eq_num -= _num_eqs_balanced(n, l, l)
    if eq_num < bell_same_shape(n):
        rhs_shape = lhs_shape
        # Slow code here
        for rhyme in all_rhymes(n + 1):
            flipped = rhyme[(n // 2) + 1:] + rhyme[:(n // 2) + 1]
            if canonicalize_rhyme(flipped) < rhyme:
                continue
            if rhyme == flipped and n > 0:
                continue
            if eq_num == 0:
                break
            eq_num -= 1
    else:
        eq_num -= bell_same_shape(n)
        shape_diff, pid = divmod(eq_num, bell(n + 2))
        rhs_shape = shape_from_id(n // 2, l + 1 + shape_diff)
        rhyme = get_rhyme_by_id(n + 1, pid)
    return Equation(lhs_shape, rhs_shape, rhyme)





##### Code used when the module is used as a script


def process_equation(eq_str: str) -> None:
    """Process a given equation, printing its id and canonical form."""
    eq_str = eq_str.strip("[,]")
    if eq_str.startswith("*"):
        dual = True
        eq_str = eq_str[1:]
    else:
        dual = False
    try:
        input_eq = int(eq_str)
    except ValueError:
        input_eq = None
    if isinstance(input_eq, int):
        eq = Equation.from_id(input_eq)
        if dual:
            eq = eq.dual()
            eq_num = eq.id
            print(f"The dual of Equation {input_eq} is Equation {eq_num}: {eq}")
        else:
            print(f"Equation {input_eq}: {eq}")
    else:
        input_eq = Equation.from_str(eq_str)
        if dual:
            dual_eq = input_eq.dual()
            dual_num = dual_eq.id
            if dual_num == 0:
                print(f"The dual of the tautological equation '{eq_str}' is: {dual_eq}")
            else:
                print(f"The dual of '{eq_str}' is Equation {dual_num}: {dual_eq}")
        else:
            eq_num = input_eq.id
            if eq_num == 0:
                print(f"The tautological equation '{eq_str}' is: {input_eq}")
            else:
                print(f"The equation '{eq_str}' is Equation {eq_num}: {input_eq}")

def main():
    """Main function to run the program."""
    parser = argparse.ArgumentParser(
        description="Canonicalize equations and find their numbers."
    )
    parser.add_argument(
        "equations",
        nargs="*",
        help="The equations to canonicalize (if not in interactive mode)",
    )
    parser.add_argument(
        "--interactive", "-i", action="store_true", help="Run in interactive mode"
    )

    args = parser.parse_args()

    if args.interactive:
        print("Welcome to the interactive equation canonicalizer!")
        print("Type 'exit' or 'quit' to end the session.")
        while True:
            eq = input("Enter an equation: ").strip()
            if eq.lower() in ["exit", "quit"]:
                print("Goodbye!")
                break
            process_equation(eq)
    else:
        if not sys.stdin.isatty():
            args.equations = [eq for l in sys.stdin for eq in l.split()] + args.equations
        if args.equations:
            for eq in args.equations:
                process_equation(eq)
        else:
            parser.print_help()


if __name__ == "__main__":
    main()


Overwriting find_equation_id.py


In [ ]:
"""
etp_oracle.py

Core "ground truth" layer for the semantic drift pipeline.
Wraps three things:
  1. Equation normalization + ETP node lookup (reuses ETP's own find_equation_id.py logic)
  2. Implication-graph oracle (fast SQLite lookup over the precomputed ETP edge list)
  3. Brute-force finite-magma oracle (fallback for equations outside the precomputed graph,
     or for spot-checking / building intuition)

Setup (one-time):
    1. Clone the ETP repo:
         git clone https://github.com/teorth/equational_theories.git
    2. Grab a snapshot of the implication graph edge list, e.g.
         equational_theories/data/2024-11-10-edge_list.csv.zip
       (check the repo's data/ folder or the Zulip "Database of implications" thread
       for the most recent snapshot available for download)
    3. Unzip it, then run build_db() below ONCE to build implications.db (~1GB, ~90 sec).
       You only do this once; every query after that is instant.

NOTE: the snapshot you download is a point-in-time export, not necessarily the fully
completed graph (~April 2025). For a handful of well-known equations (associativity,
commutativity, etc.) this makes no difference, but if a specific edge you need shows up
as unresolved, that itself is a legitimate "unknown" result worth recording -- and you
can cross-check by browsing the live Equation Explorer at
https://teorth.github.io/equational_theories/implications/?<id>
"""

import csv
import sqlite3
import time
import itertools
from pathlib import Path

from find_equation_id import Equation  # ETP's own parser/canonicalizer/id-mapper

ETP_FRAGMENT_SIZE = 4694  # number of catalogued laws (order <= 4)


# ---------------------------------------------------------------------------
# 1. Normalization + node lookup
# ---------------------------------------------------------------------------

import re

# Common LaTeX operator command names a model might use instead of the literal
# '◇' character -- extend this list if you see others in practice.
_LATEX_OP_NAMES = [
    r"\\oplus", r"\\otimes", r"\\diamond", r"\\ast", r"\\cdot",
    r"\\circ", r"\\bullet", r"\\times", r"\\star",
    r"\\triangleleft", r"\\triangleright", r"\\wedge", r"\\vee",
    r"\\odot", r"\\ominus", r"\\oslash", r"\\dagger",
]


def clean_llm_equation_output(raw: str) -> str:
    """
    Smaller/open models are far less reliable than Claude at following
    'output ONLY the equation, no formatting' instructions -- they commonly
    wrap output in LaTeX math delimiters and use LaTeX command names for the
    operator instead of the literal '◇' character. This strips that noise
    BEFORE handing the string to the real parser, so normalize_equation()
    sees plain text like 'x ◇ x = x' instead of '\\[ x \\oplus x = x \\]'.

    This is a best-effort cleanup, not a guarantee -- always check
    generated_status == 'invalid' rows manually; some model outputs are
    noisy in ways this won't catch.
    """
    s = raw.strip()

    # Strip LaTeX math delimiters: \( \), \[ \], $...$, $$...$$
    s = re.sub(r"\\\(|\\\)|\\\[|\\\]|\$\$?", "", s)

    # Strip \text{...}, \mathrm{...} wrappers some models add around variable names
    s = re.sub(r"\\(?:text|mathrm|mathbf|mathbin)\{([^}]*)\}", r"\1", s)

    # Replace LaTeX operator command names with the literal ◇ character
    for pattern in _LATEX_OP_NAMES:
        s = re.sub(pattern, "◇", s)

    # Collapse any remaining stray backslashes and extra whitespace
    s = s.replace("\\", "")
    s = re.sub(r"\s+", " ", s).strip()

    return s


def normalize_equation(eq_str: str):
    """
    Parse a raw equation string (operator can be '.', '*', or '◇'), canonicalize
    variable names/parenthesization, and map it to an ETP id if possible.

    Automatically runs clean_llm_equation_output() first, so this is safe to
    call directly on raw, possibly-noisy LLM output.

    Returns a dict with:
      status: "invalid" | "outside_fragment" | "in_fragment"
      normalized: canonical string form (or None if unparseable)
      etp_id: integer id (or None)
      error: parse error message (only if status == "invalid")
    """
    eq_str = clean_llm_equation_output(eq_str)
    try:
        eq = Equation.from_str(eq_str)
    except Exception as first_error:
        # Retry once, extracting just the equation-shaped substring in case
        # the model added surrounding prose ("The equation is: ...").
        extracted = _extract_equation_substring(eq_str)
        if extracted != eq_str:
            try:
                eq = Equation.from_str(extracted)
            except Exception:
                return {"status": "invalid", "normalized": None, "etp_id": None, "error": str(first_error)}
        else:
            return {"status": "invalid", "normalized": None, "etp_id": None, "error": str(first_error)}

    eq_id = int(eq.id)  # cast from sympy Integer; 0 means tautological (x=x style)
    normalized = str(eq)

    if eq_id == 0 or eq_id > ETP_FRAGMENT_SIZE:
        # id 0 (tautology, e.g. tautological both-sides-equal) and ids beyond the
        # 4694 catalogued (order<=4) laws both fall outside the precomputed graph
        return {"status": "outside_fragment", "normalized": normalized, "etp_id": eq_id, "error": None}

    return {"status": "in_fragment", "normalized": normalized, "etp_id": eq_id, "error": None}


def _extract_equation_substring(s: str) -> str:
    """
    Fallback for output with surrounding prose (e.g. 'The equation is: x ◇ x = x').
    Finds the longest substring that looks like a bare equation -- variables,
    the ◇ operator, parentheses, whitespace, and exactly the '=' sign(s) --
    and returns just that. Returns the original string unchanged if no clear
    equation-shaped substring is found (so the caller's original error message
    still surfaces rather than a confusing new one).
    """
    matches = re.findall(r"[a-zA-Z0-9◇()\s]+=[a-zA-Z0-9◇()=\s]+", s)
    if not matches:
        return s
    # Prefer the longest match -- most likely to be the full equation rather
    # than a fragment of it.
    return max(matches, key=len).strip()


def equation_text_by_id(etp_id: int) -> str:
    """Look up the canonical text of a catalogued equation by its ETP number."""
    return str(Equation.from_id(etp_id))


# ---------------------------------------------------------------------------
# 2. Implication-graph oracle (SQLite-backed)
# ---------------------------------------------------------------------------

def build_db(edge_list_csv: str, db_path: str = "implications.db"):
    """
    One-time setup: import the ETP edge_list.csv snapshot into an indexed SQLite db
    for O(log n) lookups instead of scanning a >1GB CSV every time.
    """
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode = OFF")
    cur.execute("PRAGMA synchronous = OFF")
    cur.execute("CREATE TABLE IF NOT EXISTS edges (source INTEGER, target INTEGER, outcome TEXT)")

    t0 = time.time()
    with open(edge_list_csv, newline="") as f:
        reader = csv.reader(f)
        next(reader)  # header
        batch, n = [], 0
        for row in reader:
            src = int(row[0].replace("Equation", ""))
            tgt = int(row[1].replace("Equation", ""))
            batch.append((src, tgt, row[2]))
            n += 1
            if len(batch) >= 200_000:
                cur.executemany("INSERT INTO edges VALUES (?,?,?)", batch)
                batch = []
        if batch:
            cur.executemany("INSERT INTO edges VALUES (?,?,?)", batch)
    conn.commit()
    cur.execute("CREATE INDEX IF NOT EXISTS idx_st ON edges(source, target)")
    conn.commit()
    conn.close()
    print(f"Built {db_path}: {n} rows in {time.time()-t0:.1f}s")


class ImplicationGraph:
    """Thin query wrapper around the indexed edge-list database."""

    TRUE_OUTCOMES = {"implicit_proof_true", "explicit_proof_true"}
    FALSE_OUTCOMES = {"implicit_proof_false", "explicit_proof_false"}

    def __init__(self, db_path: str = "implications.db"):
        self.conn = sqlite3.connect(db_path)

    def _edge(self, src: int, tgt: int):
        cur = self.conn.execute(
            "SELECT outcome FROM edges WHERE source=? AND target=?", (src, tgt)
        )
        row = cur.fetchone()
        return row[0] if row else None

    def implies(self, src: int, tgt: int):
        """True / False / None (unresolved in this snapshot) for src => tgt."""
        outcome = self._edge(src, tgt)
        if outcome in self.TRUE_OUTCOMES:
            return True
        if outcome in self.FALSE_OUTCOMES:
            return False
        return None  # missing edge or an "unknown"-flavoured outcome string

    def relation(self, intended_id: int, generated_id: int):
        """
        Compare two ETP ids and return one of:
        'equivalent' | 'stronger' | 'weaker' | 'incomparable' | 'unknown'

        Convention: "stronger"/"weaker" is stated relative to the INTENDED equation,
        matching your dataset's semantics (generated is stronger => generated implies
        intended but not vice versa, i.e. generated adds constraints).
        """
        gen_implies_intended = self.implies(generated_id, intended_id)
        intended_implies_gen = self.implies(intended_id, generated_id)

        if gen_implies_intended is None or intended_implies_gen is None:
            return "unknown"
        if gen_implies_intended and intended_implies_gen:
            return "equivalent"
        if gen_implies_intended and not intended_implies_gen:
            return "stronger"
        if intended_implies_gen and not gen_implies_intended:
            return "weaker"
        return "incomparable"


# ---------------------------------------------------------------------------
# 3. Brute-force finite-magma oracle (fallback / spot-check)
# ---------------------------------------------------------------------------

def _all_magmas(n: int):
    """Yield every possible n x n multiplication table (Cayley table) on {0..n-1}.
    Only usable for tiny n (2, 3, maybe 4) -- grows as n^(n^2)."""
    entries = list(itertools.product(range(n), repeat=n * n))
    for e in entries:
        table = [e[i * n:(i + 1) * n] for i in range(n)]
        yield table


def _eval_term(shape, rhyme_iter, assignment, table):
    if shape is None:
        return assignment[next(rhyme_iter)]
    left = _eval_term(shape[0], rhyme_iter, assignment, table)
    right = _eval_term(shape[1], rhyme_iter, assignment, table)
    return table[left][right]


def _satisfies(eq: Equation, table, n: int) -> bool:
    num_vars = eq.num_vars()
    for assignment in itertools.product(range(n), repeat=num_vars):
        rhyme_iter = iter(eq.rhyme)
        lhs_val = _eval_term(eq.lhs_shape, rhyme_iter, assignment, table)
        rhs_val = _eval_term(eq.rhs_shape, rhyme_iter, assignment, table)
        if lhs_val != rhs_val:
            return False
    return True


_satisfies_correct = _satisfies  # backwards-compatible alias


def brute_force_relation(eq_a_str: str, eq_b_str: str, max_size: int = 3):
    """
    Search all magmas of size 2..max_size for a counterexample to
    'A implies B' and to 'B implies A'.

    Returns a dict:
      a_implies_b: False (counterexample found) or None (none found up to max_size -> inconclusive)
      b_implies_a: same
      witnesses: the counterexample tables found, if any

    This can only ever PROVE non-implication (via a witness magma); it can never prove
    implication holds for ALL magmas -- absence of a counterexample up to max_size=3 or 4
    is evidence, not certainty. Sizes 2-4 catch ~96% of real-world false implications
    per the ETP paper, which is why this is a good fallback bound.
    """
    eq_a = Equation.from_str(eq_a_str)
    eq_b = Equation.from_str(eq_b_str)

    result = {"a_implies_b": None, "b_implies_a": None, "witness_a_not_b": None, "witness_b_not_a": None}

    for n in range(2, max_size + 1):
        for table in _all_magmas(n):
            a_holds = _satisfies_correct(eq_a, table, n)
            b_holds = _satisfies_correct(eq_b, table, n)
            if a_holds and not b_holds and result["a_implies_b"] is None:
                result["a_implies_b"] = False
                result["witness_a_not_b"] = table
            if b_holds and not a_holds and result["b_implies_a"] is None:
                result["b_implies_a"] = False
                result["witness_b_not_a"] = table
        if result["a_implies_b"] is False and result["b_implies_a"] is False:
            break  # already fully incomparable, no need to search bigger sizes

    return result

## Get the ETP equation graph + build the local oracle database

This is your **reference / ground truth** — the pre-verified implication graph from the Equational Theories Project. Everything in Section D (semantic validation) checks against this.

In [ ]:
!git clone --depth 1 https://github.com/teorth/equational_theories.git
!cd equational_theories/data && unzip -o 2024-11-10-edge_list.csv.zip
!cp equational_theories/data/equations.txt .


fatal: destination path 'equational_theories' already exists and is not an empty directory.
Archive:  2024-11-10-edge_list.csv.zip
  inflating: edge_list.csv           


In [ ]:
import os, shutil
if os.path.exists(DRIVE_DB_PATH):
    print('Restoring implications.db from Drive (no rebuild needed)...')
    shutil.copy(DRIVE_DB_PATH, 'implications.db')
else:
    print('Building implications.db from scratch (~90s, one-time)...')
    from etp_oracle import build_db
    build_db('equational_theories/data/edge_list.csv', db_path='implications.db')
    shutil.copy('implications.db', DRIVE_DB_PATH)
print('Ready:', os.path.getsize('implications.db')/1e6, 'MB')


Restoring implications.db from Drive (no rebuild needed)...
Ready: 1089.241088 MB


---
# Section A — Load an open model LOCALLY (not a hosted API)

Using local weights instead of a hosted API is a deliberate choice here: if you plan to do interpretability work later (probing hidden states, activation patching, steering vectors), you need **direct access to the model's internals**, which a hosted API can never give you, no matter how permissions are configured. This cell loads the model straight into the Colab GPU's memory and exposes `model` and `tokenizer` as top-level variables you can reach into from later cells.

**Model choice**: `Qwen/Qwen2.5-1.5B-Instruct` — small enough to run comfortably on Colab's free-tier T4 GPU (~15GB VRAM), no gated-access approval needed (unlike some Llama checkpoints), and capable enough for short equation/description tasks. Swap `MODEL_NAME` for something else if you outgrow it.

**If you're planning to use TransformerLens** for the probing/steering work specifically, check TransformerLens's current supported-model list before committing to Qwen -- support varies by architecture and version, and a GPT-2/Llama/Mistral/Gemma-family model may be a smoother fit depending on what version of TransformerLens you're on.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # change here if you need a different model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('Loaded', MODEL_NAME, 'on', model.device)
print('model and tokenizer are now available as top-level variables --')
print('use these directly for hooks / hidden-state access / steering later.')


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda:0
model and tokenizer are now available as top-level variables --
use these directly for hooks / hidden-state access / steering later.


### The single call_llm() function -- everything downstream goes through this

Runs generation locally (no network call, no API key, no permissions to configure). Deterministic (`do_sample=False`) so results are reproducible run to run -- useful when you're later comparing against probed/steered variants of the same model.

In [ ]:
def call_llm(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,          # <- explicitly request dict form
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,               # <- unpack input_ids AND attention_mask
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]   # <- index into the dict
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

### Sanity check: a trivial, throwaway call
Confirms the model loaded and generates sensible text before we trust it with real data.

In [ ]:
test_reply = call_llm("Reply with exactly the word: OK")
print(repr(test_reply))
print('Section A passed: local model is generating text.')


'OK'
Section A passed: local model is generating text.


---
# Section B — Component 1: Intended theory

Pick a real equation from ETP by its catalogue number. This is your ground-truth starting point -- everything downstream is measured against this exact equation.

In [ ]:
# from etp_oracle import equation_text_by_id

INTENDED_ID = 4512  # associativity -- change this to try a different equation
intended_equation = equation_text_by_id(INTENDED_ID)
print(f'Intended theory: E{INTENDED_ID} = {intended_equation}')


Intended theory: E4512 = x ◇ (y ◇ z) = (x ◇ y) ◇ z


---
# Section C — Component 2: Representation change (the single LLM call, applied)

We do the **equation -> natural language -> equation** round trip, using the SAME `call_llm()` function from Section A for both hops. This is the representation-change step the framework describes -- specifically the 'natural language to equation' direction is the one we're studying for drift, so we generate the NL description first, then translate it back blind (the translation call never sees the original equation, only the description).

In [ ]:
def describe_equation_nl(equation_str: str) -> str:
    """Representation change, direction 1: equation -> natural language."""
    prompt = (
        "Describe the following magma equation in plain, unambiguous natural "
        "language, as you would to a mathematician who cannot see the formula. "
        "Do not use the '◇' symbol or any formal notation in your description.\n\n"
        f"Equation: {equation_str}"
    )
    return call_llm(prompt, max_new_tokens=400)

def translate_nl_to_equation(nl_description: str) -> str:
    """Representation change, direction 2: natural language -> equation.
    This call ONLY sees the description -- never the original equation --
    which is what makes any mismatch meaningful.

    NOTE: smaller/open models are much less reliable than a frontier model
    at following 'output ONLY the equation' instructions -- they commonly
    wrap output in LaTeX (\\( \\), \\[ \\]) and use LaTeX command names
    (\\oplus) instead of the literal '◇' character. The prompt below is
    written to reduce that with an explicit negative example and a
    one-shot demonstration; normalize_equation() ALSO cleans this up
    automatically as a safety net, so a stray LaTeX wrapper won't
    silently fail the whole row.
    """
    prompt = (
        "Translate the following natural-language description of a law for a "
        "binary operation into a single formal equation.\n\n"
        "STRICT OUTPUT RULES:\n"
        "- Use the literal character ◇ for the operation -- never \\oplus, \\cdot, *, or any LaTeX command.\n"
        "- Do NOT wrap the equation in \\( \\), \\[ \\], $ $, or any other delimiters.\n"
        "- Do NOT add any preamble like 'The equation is:' or 'Sure, here it is:'.\n"
        "- Output ONLY the bare equation and nothing else.\n\n"
        "Example -- if the description were 'combining two elements gives the same "
        "result regardless of order', your ENTIRE response should be exactly:\n"
        "x ◇ y = y ◇ x\n\n"
        f"Description: {nl_description}\n\n"
        "Your answer (bare equation only):"
    )
    return call_llm(prompt)


In [ ]:
nl_description = describe_equation_nl(intended_equation)
print('NL description (from the LLM):')
print(' ', nl_description)

generated_equation_raw = translate_nl_to_equation(nl_description)
print('\nGenerated equation (translated back by the LLM, blind to the original):')
print(' ', generated_equation_raw)


NL description (from the LLM):
  The given equation is an example of a binary operation called "◇" applied twice on three variables. Let's break it down:

1. The left side of the equation, \( x \) ◇ \( (y \) ◇ \( z ) \), means that we first apply the operation ◇ to \( y \) and \( z \). This gives us some result, which we'll call \( w \).

2. Then, we take this result \( w \) and apply the operation ◇ again to \( x \). So, the right side of the equation, \( (x \) ◇ \( y) \) ◇ \( z \), means that we first apply the operation ◇ to \( x \) and \( y \), giving us another result, which we'll call \( v \). We then take this new result \( v \) and apply the operation ◇ to \( z \).

3. Therefore, the entire equation states that applying the operation ◇ twice in succession between two different pairs of variables results in the same outcome as applying the operation ◇ once between those two variables and then between the other pair of variables.

In simpler terms, if we have three numbers \( x \

---
# Section D — Component 3: Generated output classification

Before comparing meanings, check the basic shape of what came back:
- Did it even parse as a valid equation?
- If valid, does it match one of ETP's 4,694 catalogued laws ("in fragment"), or is it a valid equation that just isn't catalogued ("outside fragment")?

In [ ]:
generated_equation_raw

'\\( (x \\) ◇ \\( y) \\) ◇ \\( z = x \\) ◇ \\( (y \\) ◇ \\( z) \\)'

In [ ]:
# from etp_oracle import normalize_equation

norm = normalize_equation(generated_equation_raw)
print('Normalized form:', norm['normalized'])
print('Status:', norm['status'])       # invalid / outside_fragment / in_fragment
print('Matched ETP id:', norm['etp_id'])


Normalized form: x ◇ (y ◇ z) = (x ◇ y) ◇ z
Status: in_fragment
Matched ETP id: 4512


---
# Section E — Component 4: Semantic validation (2+ methods)

**Method 1 — Implication graph lookup** (primary, exact): checks the ETP graph directly. Mutual implication = equivalent; one-way = stronger/weaker; neither = incomparable.

**Method 2 — Brute-force finite magma search** (fallback / second method): used automatically when the generated equation is outside the 4,694-law fragment, so there's no graph edge to look up. Searches small magmas (size 2-3) for a concrete counterexample. Can only ever disprove, never prove -- absence of a counterexample is evidence, not certainty.

Using both satisfies the "2+ methods" requirement: graph lookup when possible, brute-force as an independent check or fallback when it isn't.

In [ ]:
from etp_oracle import ImplicationGraph, brute_force_relation

graph = ImplicationGraph('implications.db')

if norm['status'] == 'invalid':
    label = 'invalid'
    method_used = 'none (parse failure)'

elif norm['status'] == 'outside_fragment':
    # Method 2: brute-force, since there's no graph node to look up
    bf = brute_force_relation(intended_equation, norm['normalized'], max_size=3)
    method_used = 'brute_force (outside fragment)'
    if bf['a_implies_b'] is False and bf['b_implies_a'] is False:
        label = 'incomparable'
    elif bf['a_implies_b'] is False:
        label = 'weaker'
    elif bf['b_implies_a'] is False:
        label = 'stronger'
    else:
        label = 'unknown'
    print('Brute-force detail:', bf)

else:
    # Method 1: implication graph lookup (ETP reference graph)
    label = graph.relation(INTENDED_ID, norm['etp_id'])
    method_used = 'implication_graph'
    # Cross-check with Method 2 anyway, as a second independent signal:
    bf = brute_force_relation(intended_equation, norm['normalized'], max_size=2)
    print('Cross-check via brute-force (size<=2):', bf)

print()
print('=== RESULT ===')
print('Intended:  ', intended_equation)
print('Generated: ', generated_equation_raw)
print('Normalized:', norm['normalized'])
print('Method used:', method_used)
print('Label:', label)


Cross-check via brute-force (size<=2): {'a_implies_b': None, 'b_implies_a': None, 'witness_a_not_b': None, 'witness_b_not_a': None}

=== RESULT ===
Intended:   x ◇ (y ◇ z) = (x ◇ y) ◇ z
Generated:  \( (x \) ◇ \( y) \) ◇ \( z = x \) ◇ \( (y \) ◇ \( z) \)
Normalized: x ◇ (y ◇ z) = (x ◇ y) ◇ z
Method used: implication_graph
Label: equivalent


---
# Section F — Component 5: Analysis

For a single run, analysis is just reading the result above and asking: *was there drift, and if so, what kind?* Once you loop this over many equations (next section), this becomes a real summary.


In [ ]:
if label == 'equivalent':
    print('No semantic drift detected -- the translation preserved meaning.')
elif label in ('weaker', 'stronger'):
    print(f'Directional drift detected: generated equation is {label} than intended.')
elif label == 'incomparable':
    print('The generated equation is unrelated to the intended one -- a genuine mismatch,')
    print('even though it may be syntactically valid and even a real catalogued ETP law.')
elif label == 'invalid':
    print('The LLM failed to produce a parseable equation at all.')
else:
    print(f'Label: {label} -- inspect manually.')


No semantic drift detected -- the translation preserved meaning.


---
# Section G — Loop this over many equations (once Sections A-F work)

Only run this after confirming the single-equation flow above works exactly as expected. This loops Sections C-F over your full equation list.

In [ ]:
import csv

def build_explanation(intended_id, intended_eq, nl, gen_raw, norm, label, oracle_used):
    """Builds a human-readable narrative explaining this row's result."""
    lines = []
    lines.append(f"Intended theory: E{intended_id} = \"{intended_eq}\"")
    lines.append(f"NL description generated: \"{nl}\"")
    lines.append(f"LLM's translation back to a formal equation: \"{gen_raw}\"")

    if norm['status'] == 'invalid':
        lines.append("Normalization failed: the LLM's output could not be parsed as a valid equation.")
        lines.append("Semantic drift: UNRESOLVABLE -- no comparison possible, the translation step itself failed.")
        return " | ".join(lines)

    lines.append(f"Normalized form: \"{norm['normalized']}\"" +
                  (f" (matches ETP E{norm['etp_id']})" if norm['status'] == 'in_fragment' else " (not a catalogued ETP law)"))
    lines.append(f"Comparison method used: {oracle_used}")

    if label == "equivalent":
        drift = "NONE -- the LLM's translation preserved the intended meaning exactly."
    elif label == "stronger":
        drift = ("STRENGTHENING -- the generated equation implies the intended one, but not vice versa. "
                  "The LLM's translation added a constraint that was not present in the original.")
    elif label == "weaker":
        drift = ("WEAKENING -- the intended equation implies the generated one, but not vice versa. "
                  "The LLM's translation dropped a constraint that was present in the original.")
    elif label == "incomparable":
        drift = ("INCOMPARABLE -- neither equation implies the other. "
                  "The LLM's translation describes a genuinely different, unrelated law, "
                  "even though it may be syntactically valid.")
    elif label == "unknown":
        drift = "UNKNOWN -- the available oracle could not determine the relationship within its search bounds."
    else:
        drift = f"{label} (unrecognized label -- inspect manually)."

    lines.append(f"Semantic drift: {drift}")
    return " | ".join(lines)


def run_one(intended_id, graph):
    intended_eq = equation_text_by_id(intended_id)
    nl = describe_equation_nl(intended_eq)
    gen_raw = translate_nl_to_equation(nl)
    norm = normalize_equation(gen_raw)

    row = {'intended_id': intended_id, 'intended_equation': intended_eq,
           'nl_description': nl, 'generated_equation_raw': gen_raw,
           'normalized_equation': norm['normalized'], 'generated_status': norm['status'],
           'generated_etp_id': norm['etp_id'], 'oracle_used': None, 'label': None,
           'explanation': None}

    if norm['status'] == 'invalid':
        row['label'] = 'invalid'; row['oracle_used'] = 'none (parse failure)'
    elif norm['status'] == 'outside_fragment':
        bf = brute_force_relation(intended_eq, norm['normalized'], max_size=3)
        row['oracle_used'] = 'brute_force'
        if bf['a_implies_b'] is False and bf['b_implies_a'] is False: row['label']='incomparable'
        elif bf['a_implies_b'] is False: row['label']='weaker'
        elif bf['b_implies_a'] is False: row['label']='stronger'
        else: row['label']='unknown'
    else:
        row['label'] = graph.relation(intended_id, norm['etp_id'])
        row['oracle_used'] = 'implication_graph'

    row['explanation'] = build_explanation(intended_id, intended_eq, nl, gen_raw, norm, row['label'], row['oracle_used'])
    return row

# Load your equation list from equation_selection.csv if you've uploaded one,
# otherwise fall back to a small built-in sample.
import os
if os.path.exists('/content/equation_selection.csv'):
    with open('/content/equation_selection.csv') as f:
        equation_ids = [int(r['id']) for r in csv.DictReader(f)]
else:
    equation_ids = [1, 2, 43, 4512, 168]
    print('No equation_selection.csv found -- using a small built-in sample. '
          'Upload your CSV and re-run this cell to use your full list.')

results = []
for eq_id in equation_ids:
    print(f'Processing E{eq_id}...')
    r = run_one(eq_id, graph)
    results.append(r)
    print(f'  -> {r["label"]}')
    print(f'  {r["explanation"]}\n')

with open('dataset.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print(f'\nWrote {len(results)} rows to dataset.csv')

Processing E1...
  -> equivalent
  Intended theory: E1 = "x = x" | NL description generated: "The equation \( x = x \) simply states that a variable \( x \) is equal to itself. This is an identity and holds true for all values of \( x \). It represents equality without any conditions or constraints on what \( x \) can be." | LLM's translation back to a formal equation: "x = x" | Normalized form: "x = x" (matches ETP E1) | Comparison method used: implication_graph | Semantic drift: NONE -- the LLM's translation preserved the intended meaning exactly.

Processing E2...
  -> equivalent
  Intended theory: E2 = "x = y" | NL description generated: "The equation \( x = y \) simply states that the value of \( x \) is equal to the value of \( y \). This means they represent the same quantity or point on a number line." | LLM's translation back to a formal equation: "x = y" | Normalized form: "x = y" (matches ETP E2) | Comparison method used: implication_graph | Semantic drift: NONE -- the LLM's

In [ ]:
# import csv

# def run_one(intended_id, graph):
#     intended_eq = equation_text_by_id(intended_id)
#     nl = describe_equation_nl(intended_eq)
#     gen_raw = translate_nl_to_equation(nl)
#     norm = normalize_equation(gen_raw)

#     row = {'intended_id': intended_id, 'intended_equation': intended_eq,
#            'nl_description': nl, 'generated_equation_raw': gen_raw,
#            'normalized_equation': norm['normalized'], 'generated_status': norm['status'],
#            'generated_etp_id': norm['etp_id'], 'oracle_used': None, 'label': None}

#     if norm['status'] == 'invalid':
#         row['label'] = 'invalid'; row['oracle_used'] = 'none (parse failure)'
#     elif norm['status'] == 'outside_fragment':
#         bf = brute_force_relation(intended_eq, norm['normalized'], max_size=3)
#         row['oracle_used'] = 'brute_force'
#         if bf['a_implies_b'] is False and bf['b_implies_a'] is False: row['label']='incomparable'
#         elif bf['a_implies_b'] is False: row['label']='weaker'
#         elif bf['b_implies_a'] is False: row['label']='stronger'
#         else: row['label']='unknown'
#     else:
#         row['label'] = graph.relation(intended_id, norm['etp_id'])
#         row['oracle_used'] = 'implication_graph'
#     return row

# # Load your equation list from equation_selection.csv if you've uploaded one,
# # otherwise fall back to a small built-in sample.
# import os
# if os.path.exists('/content/equation_selection.csv'):
#     with open('/content/equation_selection.csv') as f:
#         equation_ids = [int(r['id']) for r in csv.DictReader(f)]
# else:
#     equation_ids = [1, 2, 43, 4512, 168]
#     print('No equation_selection.csv found -- using a small built-in sample. '
#           'Upload your CSV and re-run this cell to use your full list.')

# results = []
# for eq_id in equation_ids:
#     print(f'Processing E{eq_id}...')
#     r = run_one(eq_id, graph)
#     results.append(r)
#     print(f'  -> {r["label"]}')

# with open('dataset.csv', 'w', newline='') as f:
#     writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
#     writer.writeheader()
#     writer.writerows(results)

# print(f'\nWrote {len(results)} rows to dataset.csv')


## Download your results

In [ ]:
from google.colab import files
files.download('dataset.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>